# Data and Loss

Training data in idris-ml is typed. A `DataPoint i o ty` pairs an input `Vector i ty`
with a target `Vector o ty`. The compiler ensures your data dimensions match your model —
you can't accidentally feed 3-element inputs to a model expecting 4.

## DataPoint

The basic training record for feedforward models:

In [1]:
:t MkDataPoint

DataPoint.MkDataPoint : Vector i ty -> Vector o ty -> DataPoint i o ty


In [2]:
:doc DataPoint

record DataPoint.DataPoint : Nat -> Nat -> Type -> Type
  Totality: total
  Visibility: public export
  Constructor: MkDataPoint : Vector i ty -> Vector o ty -> DataPoint i o ty
  Projections:
    .x : DataPoint i o ty -> Vector i ty
    .y : DataPoint i o ty -> Vector o ty
  Hint: Functor (DataPoint i o)


A classification data point with 2 input features and 3 classes (one-hot encoded):

In [3]:
dp1 : DataPoint 2 3 Double
dp1 = MkDataPoint (VTensor [1.5, -2.7]) (VTensor [0, 1, 0])

In [4]:
x dp1

VTensor [STensor 1.5, STensor -2.7]


In [5]:
y dp1

VTensor [STensor 0.0, STensor 1.0, STensor 0.0]


## Building a Dataset

A dataset is a `Vect n (DataPoint i o ty)` — the number of samples `n` and
the dimensions `i`, `o` are all in the type.

Compare to PyTorch: `[(torch.tensor([1.5, -2.7]), torch.tensor([0, 1, 0])), ...]`
— same data, but nothing stops you mixing up a 2-element input with a 3-element one.

In [6]:
dataset : Vect 3 (DataPoint 2 3 Double)
dataset = [MkDataPoint (VTensor [1.5, -2.7]) (VTensor [0, 1, 0]),
  MkDataPoint (VTensor [-3.2, 4.1]) (VTensor [0, 1, 0]),
  MkDataPoint (VTensor [5.7, 0.0]) (VTensor [0, 0, 1])]

## Recurrent Data

For RNN/LSTM models, `RecurrentDataPoint` holds sequences of time steps:

In [7]:
:t MkRecurrentDataPoint

DataPoint.MkRecurrentDataPoint : List (Vector i ty) -> List (Vector o ty) -> RecurrentDataPoint i o ty


The `.xs` field is a `List (Vector i ty)` — a variable-length sequence of
input vectors, each of fixed dimension `i`. Same for `.ys` and output dimension `o`.

## Loss Functions

A loss function takes a prediction vector and a target vector of the same size
and returns a scalar. Discover what's available:

In [8]:
:t LossFunction

0 Math.LossFunction : Type -> Type


In [9]:
:t crossEntropy

Math.crossEntropy : (Num ty, (Neg ty, (Floating ty, (Fractional ty, Ord ty)))) => LossFunction ty


In [10]:
:t nllLoss

Math.nllLoss : (Neg ty, Fractional ty) => LossFunction ty


In [11]:
:t meanSquaredError

Math.meanSquaredError : (Neg ty, (Fractional ty, Floating ty)) => LossFunction ty


In [12]:
:t binaryCrossEntropy

Math.binaryCrossEntropy : (Neg ty, (Fractional ty, Floating ty)) => LossFunction ty


These map directly to PyTorch:
- `crossEntropy` = `nn.CrossEntropyLoss` (log-softmax + NLL)
- `nllLoss` = `nn.NLLLoss` (expects log-probabilities)
- `meanSquaredError` = `nn.MSELoss`
- `binaryCrossEntropy` = `nn.BCELoss`

## Computing Loss

Build a model, run a forward pass, and compute the loss on one data point.
No training yet — just the mechanics.

In [13]:
:exec do { srand 42;
  ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (OutputLayer ll));
  dblModel <- pure (toDoubleNetwork model);
  input <- pure (the (Vector 2 Double) (VTensor [1.5, -2.7]));
  target <- pure (the (Vector 3 Double) (VTensor [0, 1, 0]));
  result <- pure (forward dblModel input);
  output <- pure (snd result);
  loss <- pure (crossEntropy output target);
  putStrLn ("Output:  " ++ show output);
  putStrLn ("Loss:    " ++ show loss) }

Output:  [0.375505156037457, 2.1735352790389495, 1.466765259510899]
Loss:    4.762107921217183


This is the building block of training: forward pass produces output, loss function
measures how wrong it is. Next notebook puts it all together with an optimizer.

## Exploring the Math Module

Loss functions, activations, and linear algebra all live in `Math`.
Use `:browse` to see everything available:

In [14]:
:browse Math

0 ActivationFunction : Type -> Type
0 AggregateFunction : (Type -> Type) -> Type -> Type
0 LossFunction : Type -> Type
0 NormalizationFunction : Type -> Type
argmax : Ord ty => Vector (S n) ty -> Fin (S n)
binaryCrossEntropy : (Neg ty, (Fractional ty, Floating ty)) => LossFunction ty
binaryCrossEntropyWithLogits : (FromDouble ty, (Neg ty, (Fractional ty, (Floating ty, Ord ty)))) => LossFunction ty
bitAccuracy : List (Vector w Double) -> List (Vector w Double) -> Double
causalMaskMatrix : (FromDouble ty, Num ty) => Matrix n n ty -> Matrix n n ty
clampMinTensor : (Num ty, Ord ty) => ty -> Tensor dims ty -> Tensor dims ty
cosineSimilarity : (Floating ty, (Fractional ty, Ord ty)) => Vector n ty -> Vector n ty -> ty
countBits : Vector w Double -> Vector w Double -> (Nat, Nat)
crossEntropy : (Num ty, (Neg ty, (Floating ty, (Fractional ty, Ord ty)))) => LossFunction ty
dotProduct : Num ty => Vector n ty -> Vector n ty -> ty
flattenMatrix : Matrix m n ty -> Vector (m * n) ty
l2Norm : (Floating

Next: [04 Training](04_training.ipynb) — optimizers, the training loop,
and an end-to-end working example.